In [2]:
! pip install -q llama-index-core llama-index-readers-file llama-index-embeddings-huggingface llama-index-vector-stores-faiss llama-index-llms-groq faiss-cpu


In [3]:

from google.colab import files

uploaded = files.upload()

pdf_path = next(
    name for name in uploaded
    if name.lower().endswith(".pdf")
)

print("Uploaded PDF:", pdf_path)


Saving attention.pdf to attention.pdf
Uploaded PDF: attention.pdf


In [7]:
import os
from getpass import getpass

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq

#os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Settings.llm = Groq(
    model="openai/gpt-oss-20b",
    api_key="GroqAPIKEY" #os.environ["GROQ_API_KEY"]
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("LLM and embedding model configured")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM and embedding model configured


In [8]:

import faiss

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
)
from llama_index.vector_stores.faiss import FaissVectorStore

# 1. Load the single PDF
documents = SimpleDirectoryReader(
    input_files=[pdf_path]
).load_data()

print("Number of document pages:", len(documents))
print("PDF loading completed!")

# 2. Create FAISS index
embedding_dimension = 384

faiss_index = faiss.IndexFlatL2(
    embedding_dimension
)

vector_store = FaissVectorStore(
    faiss_index=faiss_index
)

# 3. Connect FAISS to LlamaIndex
storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# 4. Generate embeddings and store them in FAISS
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)

print("PDF indexed successfully!")
print("Vectors stored in FAISS:", faiss_index.ntotal)


Number of document pages: 15
PDF loading completed!


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/15 [00:00<?, ?it/s]

PDF indexed successfully!
Vectors stored in FAISS: 15


In [9]:

# Create the query engine
query_engine = index.as_query_engine(
    similarity_top_k=3
)

# Ask a question
question = input("Ask a question about your PDF: ")

response = query_engine.query(question)

print("\nAnswer:")
print(response)


Ask a question about your PDF: explain abount self-attention in 5 words

Answer:
Dynamic weighted context aggregation words.


In [10]:
# display

print("Retrieved source chunks:\n")

for i, source in enumerate(response.source_nodes, start=1):
    print(f"--- Source {i} ---")
    print("Similarity score:", source.score)
    print(source.node.get_content()[:1000])
    print()


Retrieved source chunks:

--- Source 1 ---
Similarity score: 0.8821865916252136
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads. Best viewed in color.
13

--- Source 2 ---
Similarity score: 1.1720529794692993
Input-Input Layer5
The
Law
will
never
be
perfect
,
b

In [11]:
# Save this FAISS index

import os

os.makedirs("faiss_storage", exist_ok=True)

storage_context.persist(
    persist_dir="faiss_storage"
)

print("Index saved to faiss_storage")


Index saved to faiss_storage


In [12]:
# Download

import shutil
from google.colab import files

shutil.make_archive(
    "faiss_storage",
    "zip",
    "faiss_storage"
)

files.download("faiss_storage.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''
Complete workflow recap
PDF Reader

Reads the PDF pages into documents.


Embedding Model

Converts document chunks and queries into vectors.


FAISS

Finds the nearest vector matches.


Groq LLM

Generates the final answer using the retrieved context.
'''